# YC Startup Radar — 2024–2026

Pipeline: **fetch → normalize → enrich → score → AI summaries → export**. All heavy logic lives in `src/yc_radar/`; this notebook orchestrates and displays.

> AI summaries use **Haiku 4.5** and require your own `ANTHROPIC_API_KEY` in `.env`. Without a key the AI columns show a placeholder and **no API call (and no charge) is made**. Results are cached to `data/processed/ai_cache.json`, so re-runs only pay for new companies.

In [ ]:
# --- Setup: resolve repo root & make src importable ---
import sys
from pathlib import Path

ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "src"))

# Load .env so ANTHROPIC_API_KEY (if present) is available to the AI step.
try:
    from dotenv import load_dotenv
    load_dotenv(ROOT / ".env")
except Exception:
    pass

DATA_RAW = ROOT / "data" / "raw"
DATA_PROCESSED = ROOT / "data" / "processed"
print("repo root:", ROOT)

## 1. Fetch YC companies (cached)

In [ ]:
from yc_radar import fetch

records = fetch.fetch_companies(cache_path=DATA_RAW / "yc_companies.json")
print(f"fetched {len(records)} raw company records")

## 2. Normalize → filter to 2024–2026 → dedup

In [ ]:
from yc_radar import normalize

df = normalize.normalize(records)
print(f"{len(df)} companies in batches 2024-2026")
df[["name", "batch", "batch_year", "industry", "status", "team_size"]].head(10)

## 3. Enrich → investability + open-source links, then score

In [ ]:
from yc_radar import enrich, score

df = enrich.add_investability(df)
df = enrich.add_links(df)
df = score.score(df)
df = df.sort_values("score", ascending=False).reset_index(drop=True)
df[["name", "industry", "status", "investability", "score"]].head(10)

## 4. AI summaries (Haiku 4.5, cached) — needs ANTHROPIC_API_KEY

Only companies missing from `ai_cache.json` are summarized. Safe to run without a key (placeholder text, no charge).

In [ ]:
from yc_radar import ai

df = ai.add_ai_summaries(df, cache_path=DATA_PROCESSED / "ai_cache.json")
df[["name", "ai_summary", "ai_risk_notes"]].head(5)

## 4b. Merge personal annotations (rating / watchlist / notes)

Your notes live in `data/user_data.csv` keyed by slug and survive refreshes.

In [ ]:
from yc_radar import user_data

df = user_data.merge_user_data(df, path=ROOT / "data" / "user_data.csv")
print("annotation columns:", [c for c in ('my_rating','watchlist','my_notes') if c in df.columns])

## 5. Export → Parquet + CSV + styled Excel

In [ ]:
from yc_radar import export

paths = export.export(df, out_dir=DATA_PROCESSED)
for kind, p in paths.items():
    print(f"{kind:8} -> {p}")

## 6. Analytics — distributions

Quick read on the 2024–2026 cohort: which industries, batches, statuses, and regions dominate.

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(2, 2, figsize=(14, 9))

df['industry'].value_counts().head(15).plot.barh(ax=axes[0, 0], color='#4C78A8')
axes[0, 0].set_title('Companies by industry (top 15)'); axes[0, 0].invert_yaxis()

df['batch_year'].value_counts().sort_index().plot.bar(ax=axes[0, 1], color='#72B7B2')
axes[0, 1].set_title('Companies by batch year'); axes[0, 1].tick_params(axis='x', rotation=0)

df['status'].value_counts().plot.bar(ax=axes[1, 0], color='#F58518')
axes[1, 0].set_title('Companies by status'); axes[1, 0].tick_params(axis='x', rotation=30)

df['region'].replace('', 'Unknown').value_counts().head(10).plot.barh(ax=axes[1, 1], color='#54A24B')
axes[1, 1].set_title('Companies by region (top 10)'); axes[1, 1].invert_yaxis()

plt.tight_layout(); plt.show()

**Takeaways.** Industry mix shows where YC is concentrating this cohort; the batch-year bars confirm 2024–2026 coverage; status skews heavily to *Active* (most are too young to have exited); region distribution highlights geographic clustering. Sort the exported Excel or use the Streamlit dashboard by `score` to surface the most interesting companies within any of these slices.